# MTMC 11c \u2014 WILDTRACK Stages 4-5 (Association + Evaluation)

Cross-camera person association and evaluation on WILDTRACK.

**Prerequisite**: attach **11b's output** as a data source:
`Add Data -> Kernel Output -> search "mtmc-11b-wildtrack-stage-3" -> add`

**This is the iteration loop** \u2014 edit the tuning params cell and re-run.

| Stage | What | Time |
|---|---|---|
| 4 | Cross-camera association (AQE + graph clustering) | ~2 min |
| 5 | Evaluation: IDF1, MOTA, HOTA | ~1 min |

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _major, _minor = _cap.strip().split(".")
        _sm = int(_major) * 10 + int(_minor)
        if _sm < 70:
            print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) \u2014 installing compatible PyTorch ...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu124",
            ])
            print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

## 1. Clone Repo & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "faiss-cpu>=1.7", "omegaconf>=2.3", "networkx>=3.1",
    "scipy", "scikit-learn", "click", "tqdm"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "motmetrics", "lap"])
print("\u2713 Dependencies installed")

## 2. Load Checkpoint from 11b

In [ ]:
DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)

INPUT_11B = None
for candidate in [
    Path("/kaggle/input/mtmc-11b-wildtrack-stage-3"),
    Path("/kaggle/input/mtmc-11b-wildtrack-stage-3-faiss-indexing"),
]:
    if candidate.exists():
        INPUT_11B = candidate
        break
if INPUT_11B is None:
    for d in Path("/kaggle/input").iterdir():
        if (d / "checkpoint.tar.gz").exists():
            INPUT_11B = d
            break
if INPUT_11B is None:
    raise FileNotFoundError("11b output not found in /kaggle/input/")

ckpt = INPUT_11B / "checkpoint.tar.gz"
print(f"Checkpoint: {ckpt} ({ckpt.stat().st_size/1024**2:.1f} MB)")

with tarfile.open(str(ckpt), "r:gz") as tar:
    tar.extractall(path=str(DATA_OUT))

with open(DATA_OUT / "run_metadata.json") as f:
    meta = json.load(f)
RUN_NAME = meta["run_name"]
print(f"\u2713 Run: {RUN_NAME}")

run_dir = DATA_OUT / RUN_NAME
for stage in ["stage1", "stage2", "stage3"]:
    sd = run_dir / stage
    if sd.exists():
        nf = sum(1 for _ in sd.rglob("*") if _.is_file())
        print(f"  {stage}: {nf} files")

gt_dir = DATA_OUT / "gt_annotations"
GT_DIR = str(gt_dir) if gt_dir.exists() else ""
print(f"GT: {GT_DIR or 'NOT FOUND'}")

## 3. Tuning Parameters

**Edit these values** then re-run the cells below. No need to re-run 11a or 11b.

In [ ]:
# ============================================================
# Person Pipeline (WILDTRACK) -- Tuning Parameters
#
# WILDTRACK: 7 cameras, overlapping FOV, ~20 people, 2 fps
# Best local: IDF1=0.368, MTMC_IDF1=0.233, MOTA=0.118
# ============================================================

# Stage 4: Association
SIM_THRESH        = 0.30    # lower than vehicle (more overlap)
ALGORITHM         = "community_detection"
LOUVAIN_RES       = 1.5
BRIDGE_PRUNE      = 0.02
MAX_COMP_SIZE     = 40

AQE_K             = 5
AQE_ALPHA         = 5.0

APPEARANCE_WEIGHT = 0.80
HSV_WEIGHT        = 0.10
ST_WEIGHT         = 0.10

FIC_REG           = 0.10
FIC_ENABLED       = True

RERANKING         = True
RERANKING_K1      = 25
RERANKING_K2      = 8
RERANKING_LAMBDA  = 0.35

GALLERY_THRESH    = 0.35
GALLERY_ROUNDS    = 3

INTRA_MERGE       = True
INTRA_MERGE_THRESH = 0.70
INTRA_MERGE_GAP   = 40.0

# Stage 5
MTMC_ONLY         = False

## 4. Run Stages 4-5

In [ ]:
os.chdir(str(PROJECT))

cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/wildtrack.yaml",
    "--stages", "4,5",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", f"stage4.association.query_expansion.k={AQE_K}",
    "--override", f"stage4.association.query_expansion.alpha={AQE_ALPHA}",
    "--override", f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
    "--override", f"stage4.association.graph.algorithm={ALGORITHM}",
    "--override", f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
    "--override", f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
    "--override", f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
    "--override", f"stage4.association.weights.person.appearance={APPEARANCE_WEIGHT}",
    "--override", f"stage4.association.weights.person.hsv={HSV_WEIGHT}",
    "--override", f"stage4.association.weights.person.spatiotemporal={ST_WEIGHT}",
    "--override", f"stage4.association.fic.enabled={str(FIC_ENABLED).lower()}",
    "--override", f"stage4.association.fic.regularisation={FIC_REG}",
    "--override", f"stage4.association.reranking.enabled={str(RERANKING).lower()}",
    "--override", f"stage4.association.reranking.k1={RERANKING_K1}",
    "--override", f"stage4.association.reranking.k2={RERANKING_K2}",
    "--override", f"stage4.association.reranking.lambda_value={RERANKING_LAMBDA}",
    "--override", f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
    "--override", f"stage4.association.gallery_expansion.rounds={GALLERY_ROUNDS}",
    "--override", f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
    "--override", f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
    "--override", f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
    "--override", f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
]
if GT_DIR:
    cmd += ["--override", f"stage5.ground_truth_dir={GT_DIR}"]

print(f"CMD: {' '.join(str(c) for c in cmd[:10])}...")
print("=" * 70)
t0 = time.time()
result = subprocess.run(cmd, cwd=str(PROJECT))
elapsed = time.time() - t0
print("=" * 70)
print(f"\nDone in {elapsed:.1f}s (exit={result.returncode})")

## 5. Results

In [ ]:
report_path = DATA_OUT / RUN_NAME / "stage5" / "evaluation_report.json"
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)

    m = report.get("metrics", report)
    print("=" * 60)
    print("WILDTRACK PERSON PIPELINE RESULTS")
    print("=" * 60)
    print(f"  IDF1:      {m.get('idf1', m.get('IDF1', 0)):.4f}")
    print(f"  MTMC_IDF1: {m.get('mtmc_idf1', 0):.4f}")
    print(f"  MOTA:      {m.get('MOTA', m.get('mota', 0)):.4f}")
    print(f"  HOTA:      {m.get('HOTA', m.get('hota', 0)):.4f}")

    details = report.get("details", {})
    if details:
        print(f"\n  MTMC_MOTA: {details.get('mtmc_mota', 'N/A')}")
        print(f"  ID_sw:     {details.get('id_switches', 'N/A')}")
        print(f"  Tracklets: {details.get('num_tracklets', 'N/A')}")

    # Per-camera breakdown
    per_cam = report.get("per_camera", {})
    if per_cam:
        print(f"\nPer-camera IDF1:")
        for cam, cm in sorted(per_cam.items()):
            idf1 = cm.get("idf1", cm.get("IDF1", 0))
            print(f"  {cam}: {idf1:.3f}")
else:
    print(f"No evaluation report found at {report_path}")
    # Check if stage4/5 output exists
    for sn in ["stage4", "stage5"]:
        sd = DATA_OUT / RUN_NAME / sn
        if sd.exists():
            print(f"  {sn} output exists ({sum(1 for _ in sd.rglob('*'))} files)")